# Hands-on Lab: Generative AI for Data Pipelines and ETL Workflows

## Scenario
You are a data engineer for a healthcare company and have been tasked with creating a data pipeline that will capture the recorded information about liver disease parameters for patients, available in a CSV format in the hospital records, along with the prognosis of whether the said patient had a liver disease or not. You are further required to transform all attributes of the data set into numerical attributes and load the final data into an SQL database of the data science team to pick up the data for further processing.

## Objectives
In this lab, you will learn how to use generative AI to create Python codes that can:
- Extract information from a CSV file available on a URL
- Transform the data into a preferred format
- Load the data as a table to an SQL database

## Data set
For the purpose of this lab, we are making use of the Indian Liver Patient Dataset, publically available under the Creative Commons Attribution 4.0 International (CC BY 4.0) license. You may refer to the data set webpage for more details on the attributes.

In [7]:
!pip install pandas requests scikit-learn

  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 42.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 47.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 44.0 MB/s  0:00:00
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 39.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [scikit-learn] [scikit-learn]]


In [9]:
import io
import pandas as pd
import requests
import sqlite3
import subprocess
from sklearn.preprocessing import LabelEncoder

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0273EN-SkillsNetwork/labs/v1/m2/data/Indian%20Liver%20Patient%20Dataset%20%28ILPD%29.csv"

def get_content(url):
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        return resp.text
    except Exception as e:
        print(f"Requests failed with error: {e}")
        print("Attempting fallback to system curl...")
        try:
            # Fallback to curl
            result = subprocess.run(['curl', '-L', url], capture_output=True, text=True, check=True)
            return result.stdout
        except subprocess.CalledProcessError as cpe:
            print(f"Curl also failed: {cpe}")
            raise e

def preprocess_data(df):
    categorical_cols = df.select_dtypes(include=['object']).columns
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    print(df.head())
    print(label_encoders)

    return df, label_encoders

def save_to_sqlite(df):
    db_path = 'Patient_record.db'
    table_name = 'Liver_patients'
    with sqlite3.connect(db_path) as conn:
        df.to_sql(name=table_name, con=conn, if_exists="replace", index=False)
        preview = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 2", conn)
        print("SQLite table preview:")
        print(preview)

In [10]:
try:
    csv_content = get_content(url)
    df = pd.read_csv(io.StringIO(csv_content))
    print(df.head())
except Exception as e:
    print("Error fetching or reading CSV:", e)


   Age  Gender  Total_Bilirubin  Direct_Bilirubin  Alkaline_Phosphotase  \
0   65  Female              0.7               0.1                   187   
1   62    Male             10.9               5.5                   699   
2   62    Male              7.3               4.1                   490   
3   58    Male              1.0               0.4                   182   
4   72    Male              3.9               2.0                   195   

   Alamine_Aminotransferase  Aspartate_Aminotransferase  Total_Proteins  \
0                        16                          18             6.8   
1                        64                         100             7.5   
2                        60                          68             7.0   
3                        14                          20             6.8   
4                        27                          59             7.3   

   Albumin  Albumin and Globulin Ratio  Selector  
0      3.3                        0.90         

In [11]:
df, label_encoders = preprocess_data(df)

   Age  Gender  Total_Bilirubin  Direct_Bilirubin  Alkaline_Phosphotase  \
0   65       0              0.7               0.1                   187   
1   62       1             10.9               5.5                   699   
2   62       1              7.3               4.1                   490   
3   58       1              1.0               0.4                   182   
4   72       1              3.9               2.0                   195   

   Alamine_Aminotransferase  Aspartate_Aminotransferase  Total_Proteins  \
0                        16                          18             6.8   
1                        64                         100             7.5   
2                        60                          68             7.0   
3                        14                          20             6.8   
4                        27                          59             7.3   

   Albumin  Albumin and Globulin Ratio  Selector  
0      3.3                        0.90         

In [12]:
save_to_sqlite(df)

SQLite table preview:
   Age  Gender  Total_Bilirubin  Direct_Bilirubin  Alkaline_Phosphotase  \
0   65       0              0.7               0.1                   187   
1   62       1             10.9               5.5                   699   

   Alamine_Aminotransferase  Aspartate_Aminotransferase  Total_Proteins  \
0                        16                          18             6.8   
1                        64                         100             7.5   

   Albumin  Albumin and Globulin Ratio  Selector  
0      3.3                        0.90         1  
1      3.2                        0.74         1  
